        # 04 長期的な関係継続（Stage 2：Relationship Continuation）

        成立したペアの関係が時間とともに続くかを、**教育用の架空モデル**で時系列シミュレーションします（RQ4）。

        > ⚠️ **このモデルは実際の離婚確率・婚姻の継続を予測するものではありません。** 係数（alpha・beta・gamma・baseline_hazard）はすべて**架空の値**で、現実のデータから推定したものでも、実証された因果効果でもありません。「初期適合度が高いと続きやすい」という関係は、**モデルの仮定として与えたもの**です。

> **教育用の架空データ・架空モデル**です。参加者は実在の人物ではなく、Group A・B はアルゴリズム上の区分で性別や性的指向を意味しません。希望スコアは実際の結婚適性を表さず、関係継続モデルの係数は実際の離婚率や実証された因果効果ではありません。

## 準備：実験環境（最初に必ず実行）

F3 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F3 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F3 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F3 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f3-matching-algorithms"     # リポジトリの中の F3 フォルダ（ルートが F3 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f3_root(p):
    return (Path(p) / "matching_algorithms.py").exists() and (Path(p) / "relationship_stability.py").exists()


def find_f3_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F3 のルート
        if is_f3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f3_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f3_upload")
    for hit in glob.glob("/content/f3_upload/**/matching_algorithms.py", recursive=True):
        if is_f3_root(Path(hit).parent):
            return Path(hit).parent
    return None


F3_ROOT = find_f3_root()
if F3_ROOT is None:
    raise RuntimeError("F3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F3_ROOT)
if str(F3_ROOT) not in sys.path:
    sys.path.insert(0, str(F3_ROOT))
if not (F3_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。preferences.py・matching_algorithms.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import participants as pt
import preferences as pr
import matching_algorithms as ma
import relationship_stability as rs
import evaluation as ev
import matching_charts as mc

mc.setup_style()                       # 文字を大きくするなら mc.setup_style(1.3)
groups = pt.load_participants()
scores = pr.score_table(groups)
rankings = pr.build_rankings(groups, scores)
params = rs.load_parameters()          # 関係継続モデルのパラメータ（すべて架空）
A_IDS = [p["id"] for p in groups["A"]]
B_IDS = [p["id"] for p in groups["B"]]
N_TOTAL = pt.total_participants(groups)
compat = lambda a, b: pr.pair_compatibility(a, b, scores)

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", mc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F3_ROOT:", F3_ROOT)
print(f"参加者 Group A {len(A_IDS)} 人・Group B {len(B_IDS)} 人 / 関係継続モデル（架空）:",
      {k: params[k] for k in ("baseline_hazard", "alpha", "beta", "gamma", "default_annual_stress",
                               "default_conflict_resolution", "simulation_years", "random_seed")})

## 1. 年次関係解消確率

$$ p = \mathrm{clip}(\text{baseline\_hazard} - \alpha \cdot \text{compatibility} - \beta \cdot \text{conflict\_resolution} + \gamma \cdot \text{stress},\ 0,\ 1) $$

| 変数（0〜1） | 意味（架空） |
| --- | --- |
| 初期適合度 compatibility | ペアの 2 方向の希望スコアの平均 |
| 葛藤対処 conflict_resolution | 架空の指標。Core では全ペア共通の値（実在の人物について推定しない） |
| 年次ストレス stress | 架空。Core では毎年一定 |
| baseline_hazard | 基準となる年次解消確率 |

価値観・生活スタイルの一致度は、すでに初期適合度に含まれているので、**解消確率に別に加えません**（二重に数えないため）。

In [ ]:
print({k: params[k] for k in ("baseline_hazard", "alpha", "beta", "gamma", "default_annual_stress", "default_conflict_resolution")})
for c in [0.4, 0.6, 0.8, 1.0]:
    p = rs.annual_dissolution_probability(c, params["default_conflict_resolution"], params["default_annual_stress"], params)
    print(f"初期適合度 {c}: 年次解消確率 p = {p:.3f}  → 理論上の継続確率 5 年 {rs.theoretical_survival(p, 5):.1%} / "
          f"10 年 {rs.theoretical_survival(p, 10):.1%} / 20 年 {rs.theoretical_survival(p, 20):.1%}")

## 2. 確率的な継続判定

毎年 0〜1 の一様乱数 $u$ を引き、$u < p$ ならその年に解消（以後は解消のまま）。観測期間の最後まで解消しなければ**打ち切り（censored）**＝「観測期間内には解消しなかった」で、その後のことは推測しません。

In [ ]:
example = [{"pair_id": "X-Y", "participant_ids": ["X", "Y"], "algorithm": "demo", "initial_compatibility": 0.7,
            "initial_satisfaction": None}]
for r in range(5):
    rec = rs.simulate_pairs(example, params, replicate=r)[0]
    print(f"replicate {r}: dissolved={rec['dissolved']}, year={rec['dissolution_year']}, "
          f"5y={rec['continuing_5y']}, 10y={rec['continuing_10y']}, 20y={rec['continuing_20y']}, censored={rec['censored']}")

## 3. 年数と継続率：シミュレーションと理論値

解消確率が一定 $p$ なら、$T$ 年後の継続確率は理論上 $(1-p)^T$。多数のペアでシミュレーションすると理論値（破線）に近づきます。

In [ ]:
#@title グラフを表示（コードは隠しています）
N_PAIRS = 2000                          # TODO：数を変えて、理論値との差を調べよう
YEARS = params["simulation_years"]
curves, theory = {}, {}
for c in [0.4, 0.7, 1.0]:
    pairs = [{"pair_id": f"P{i}", "participant_ids": [f"a{i}", f"b{i}"], "algorithm": "demo",
              "initial_compatibility": c, "initial_satisfaction": None} for i in range(N_PAIRS)]
    recs = rs.simulate_pairs(pairs, params)
    label = f"compatibility {c}"
    curves[label] = ev.survival_curve(recs)
    p = rs.annual_dissolution_probability(c, params["default_conflict_resolution"], params["default_annual_stress"], params)
    theory[label] = [rs.theoretical_survival(p, t) for t in range(YEARS + 1)]
show(mc.plot_survival(curves, f"Continuation by initial compatibility (model, {N_PAIRS} pairs each)",
                      theory=theory), "04_by_compatibility")

## 4. 年次ストレス別の継続率

In [ ]:
#@title グラフを表示（コードは隠しています）
curves = {}
pairs = [{"pair_id": f"P{i}", "participant_ids": [f"a{i}", f"b{i}"], "algorithm": "demo",
          "initial_compatibility": 0.7, "initial_satisfaction": None} for i in range(N_PAIRS)]
for stress in [0.0, 0.3, 0.6, 1.0]:
    curves[f"stress {stress}"] = ev.survival_curve(rs.simulate_pairs(pairs, params, annual_stress=stress))
show(mc.plot_survival(curves, "Continuation by annual stress (model, compatibility 0.7)"), "04_by_stress")

## 5. 実際に成立したペア（Gale–Shapley）に適用する

5 年・10 年・20 年の継続状態を、**反復**して平均します。ペアが 4 組しかないので、1 回の結果は大きくばらつきます。

In [ ]:
REPLICATES = 500                        # TODO：反復回数を変えてみよう
gs = ma.gale_shapley(rankings, A_IDS, B_IDS)
sat = ev.satisfaction_by_participant(gs, rankings)
pairs = rs.make_pairs(gs, "gale_shapley", compat, sat)
runs = rs.monte_carlo(pairs, params, REPLICATES)
table(pairs, [("pair_id", "pair", "{}"), ("initial_compatibility", "compatibility", "{:.3f}"),
              ("initial_satisfaction", "satisfaction", "{:.2f}")])
for t in (5, 10, 20):
    m, sd = ev.mean_over_replicates(runs, lambda rec: ev.continuation_rate(rec, t))
    print(f"{t:>2} 年継続率：平均 {m:.1%}（反復間の標準偏差 {sd:.1%}、反復 {REPLICATES} 回）")
print("1 回目の反復の記録：")
table(runs[0], [("pair_id", "pair", "{}"), ("annual_dissolution_probability", "p", "{:.3f}"),
                ("dissolved", "dissolved", "{}"), ("dissolution_year", "year", "{}"),
                ("years_continued", "years", "{}"), ("continuing_5y", "5y", "{}"),
                ("continuing_10y", "10y", "{}"), ("continuing_20y", "20y", "{}"), ("censored", "censored", "{}")])

## 6. 継続期間の分布

解消したペア（年ごとの数）と、観測終了まで継続したペア（打ち切り、斜線）を**分けて**示します。

In [ ]:
#@title グラフを表示（コードは隠しています）
all_records = [r for run in runs for r in run]
show(mc.plot_duration_hist(all_records, params["simulation_years"],
                           f"Year of dissolution (Gale-Shapley pairs, {REPLICATES} replicates, model)"), "04_duration")
print(ev.duration_summary(all_records))

## 学生向け TODO

- [ ] 初期適合度の影響係数 `alpha` を変更する（`rs.with_changes(params, alpha=...)`）
- [ ] 基準解消確率 `baseline_hazard` を変更する
- [ ] 年次ストレスを変更する（年ごとに変わるストレスは Advanced B）
- [ ] 試行回数（`N_PAIRS`・`REPLICATES`）を変更し、結果のばらつきを調べる
- [ ] 係数が大きすぎると式の値が負になり `clip` で 0 になる（継続率 100%）。そうならない範囲を式から求める

## 考察（チームで記入）

- **TODO**：初期適合度と継続率の関係（RQ4）。**それがモデルの仮定から来ていること**を説明に含める
- **TODO**：反復回数・ペア数が少ないと、結果はどのくらいばらつくか

## 参考文献

- D. Gale, L. S. Shapley, "College Admissions and the Stability of Marriage", *The American Mathematical Monthly*, 69(1), 9–15, 1962.
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 線形探索・ハッシュ法、第4章 キュー、第6章 挿入ソート）
- F3 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_participants.json`、`data/relationship_parameters.json`（この教材のための**架空のデータ**）